### RAG Retrieval Pipeline

In [ ]:
# import required packages
import os
from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

load_dotenv()

In [ ]:
persistent_directory = "../db/chroma_db"

# Loading embedding model
embedding = OllamaEmbeddings(
    base_url=os.environ.get("OLLAMA_BASE_URL"),
    model="qwen3-embedding:0.6b",
)

# Loading vector database
db = Chroma(
    persist_directory=persistent_directory,
    embedding_function=embedding,
    collection_metadata={"hnsw:space": "cosine"},
)

In [ ]:
# Search for relevant documents
query = "What was NVIDIA's first graphics accelerator called?"

retriever = db.as_retriever(search_kwargs={"k": 3})

relevant_docs = retriever.invoke(query)

print(f"User Query: {query}")
print(f"{'-' * 10}-Context-{'-' * 10}")
for i, doc in enumerate(relevant_docs, 1):
    print(f"Document {i}:\n{doc.page_content}\n")


#### Testing retrieval document quality

In [ ]:
from openai import OpenAI

llm = OpenAI(base_url="http://127.0.0.1:1234/v1")
model = "nvidia/nemotron-3-nano-4b"

In [ ]:
SYSTEM_PROMPT = """I'm testing my RAG system. I will give you the following information:
User Query:
Context:

I want you to tell me if the context is good and has the answer to the user's question/query in 1-2 lines."""

In [ ]:
def test_retrieval_quality(query: str) -> str | None:
    retriever = db.as_retriever(search_kwargs={"k": 3})
    relevant_docs = retriever.invoke(query)

    context: str = ""

    for i, doc in enumerate(relevant_docs, 1):
        context += f"Document {i}:\n{doc.page_content}\n"

    response = llm.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"User Query: {query}, Context: {context}"},
        ],
    )

    return response.choices[0].message.content

print(test_retrieval_quality(query = "What was NVIDIA's first graphics accelerator called?"))

In [ ]:
# Synthetic Questions:
queries: list[str] = [
    "What was NVIDIA's first graphics accelerator called?",
    "Which company did NVIDIA acquire to enter the mobile processor market?",
    "What was Microsoft's first hardware product release?",
    "How much did Microsoft pay to acquire GitHub?",
    "In what year did Tesla begin production of the Roadster?",
    "Who succeeded Ze'ev Drori as CEO in October 2008?",
    "What was the name of the autonomous spaceport drone ship that achieved the first successful sea landing?",
    "What was the original name of Microsoft before it became Microsoft?",
]

for i, query in enumerate(queries):
    print(f"{i}. Query: {query}")
    print(f"Response: {test_retrieval_quality(query=query)}")
    print("-" * 50)